# Push and pull lab: two routes to the same experience

Push and pull share committed records but express different decisions. Push records a policy's selection before execution. Pull records the proposer's own bounded investigation during execution.

This lab inspects both channels and then reconstructs the canonical observation ledger used by M3.4.

In [1]:
from pathlib import Path
import runpy

from meta_evolve.application.derivation import observation_ledger

path = Path("main.py")
if not path.exists():
    path = Path("examples/03_experience_reuse/main.py")
demo = runpy.run_path(path)
results = demo["run_experiment"]()
print("lab ready")

lab ready


## 1. Inspect pushed bundles

The demo's `FailedAncestors` policy pushes only failed ancestor evaluations. Notice how the number of selected failures grows as the chain advances.

In [2]:
push_run = results["push-only"][0]
push_projection = push_run._projection()

for index, selected in enumerate(push_projection.context_selections, start=1):
    records = selected.bundle.records
    labels = tuple(
        f"{record.ref.kind}@{record.logical_step}" for record in records
    )
    print(
        f"trial={index}",
        f"selected={labels}",
        f"chars={selected.bundle.usage.chars}",
    )

trial=1 selected=() chars=0
trial=2 selected=('evaluation@1',) chars=1098
trial=3 selected=('evaluation@1', 'evaluation@2') chars=2197


In [3]:
bundle = push_projection.context_selections[-1].bundle
print("reason:", bundle.reason)
print("truncated:", bundle.truncated)
print("usage:", bundle.usage)
print("refs:", tuple(
    f"{record.ref.kind}@{record.logical_step}" for record in bundle.records
))

reason: failed ancestor evaluations
truncated: False
usage: ContextUsage(records=2, chars=2197)
refs: ('evaluation@1', 'evaluation@2')


A bundle is a snapshot, not a live graph handle. Replay validates its references and exact values against prior committed records without invoking the policy again.

## 2. Inspect pull grants and operations

Each successor gets a new grant with `as_of = logical_step - 1`. The final proposer searches two failed evaluations, opens both, and compares their trials.

In [4]:
pull_run = results["pull-only"][0]
pull_projection = pull_run._projection()

for issued in pull_projection.experience_grants:
    operations = pull_projection.operations_for(issued.grant.trial_id)
    print(f"grant as_of={issued.grant.as_of}")
    for attempt, resolution in operations:
        result = getattr(resolution, "result", None)
        count = len(result.references) if result is not None else 0
        print(
            f"  {attempt.request.operation:<7}",
            f"{type(resolution).__name__:<28}",
            f"refs={count}",
        )

grant as_of=0
  search  ExperienceOperationSucceeded refs=0
grant as_of=1
  search  ExperienceOperationSucceeded refs=1
  open    ExperienceOperationSucceeded refs=1
grant as_of=2
  search  ExperienceOperationSucceeded refs=2
  open    ExperienceOperationSucceeded refs=1
  open    ExperienceOperationSucceeded refs=1
  compare ExperienceOperationSucceeded refs=2


## 3. Reconstruct the push-then-pull ledger

The ledger is ordered by committed observation: bundle references first, then successful pull results in operation order. References and structured record references are deduplicated by first occurrence.

In [5]:
combined_run = results["combined"][0]
combined = combined_run._projection()
final = combined.completed_trials[-1].trial.id
proposal = combined.proposal_for(final)
ledger = observation_ledger(combined, final)

print("ledger kinds:", tuple(reference.kind for reference in ledger))
print("declared kinds:", tuple(ref.kind for ref in proposal.derived_from))
positions = tuple(ledger.index(ref) for ref in proposal.derived_from)
print("declared positions:", positions)
print("ordered subsequence:", positions == tuple(sorted(positions)))

ledger kinds: ('evaluation', 'evaluation', 'trial', 'trial', 'decision', 'decision', 'decision_explanation', 'decision_explanation', 'proposal', 'proposal', 'artifact', 'artifact', 'outcome', 'outcome')
declared kinds: ('evaluation', 'evaluation', 'trial', 'trial')
declared positions: (0, 1, 2, 3)
ordered subsequence: True


## 4. Compare channel-specific usage

Memory access is measured but does not consume task or run budgets. Context usage counts selected records and rendered characters. Pull usage additionally counts attempts and returned references.

In [6]:
for name, (result, _) in results.items():
    usage = result.experience_usage()
    print(
        f"{name:<10}",
        f"context=({usage.context.records}, {usage.context.chars})",
        f"pull=({usage.pull.operations}, {usage.pull.results},",
        f"{usage.pull.records}, {usage.pull.chars})",
    )

no-memory  context=(0, 0) pull=(0, 0, 0, 0)
push-only  context=(3, 3295) pull=(0, 0, 0, 0)
pull-only  context=(0, 0) pull=(7, 8, 20, 20559)
combined   context=(3, 3295) pull=(7, 8, 20, 20855)


## Try next

Change `FailedAncestors` to push only one failure, lower `PullAccess.max_operations`, or remove the comparison call. Predict the changed ledger and usage before rerunning. The interesting question is not merely whether memory helps, but **which access decision helped at what cost**.